## Redact the PDF

In [7]:
# INPUT_PDF='../data/Submittal and Product Description.pdf'
# OUTPUT_PDF='../data/Submittal and Product Description_redacted.pdf'
INPUT_PDF='../data/Spec 14 24 00 - Hydraulic Elevators.pdf'
OUTPUT_PDF='../data/Spec 14 24 00 - Hydraulic Elevators_redacted.pdf'

In [ ]:
import fitz  # PyMuPDF

def redact_text_in_pdf(input_pdf, output_pdf, search_terms):
    """
    Redact specific text terms from PDF
    """
    doc = fitz.open(input_pdf)
    
    for page in doc:
        # Search for each term and redact it
        for term in search_terms:
            text_instances = page.search_for(term)
            for inst in text_instances:
                # Add redaction annotation
                page.add_redact_annot(inst, fill=(0, 0, 0))  # Black fill
        
        # Apply all redactions on this page
        page.apply_redactions()
    
    doc.save(output_pdf)
    doc.close()

# Usage
search_terms = [
    "Bonham ISD",
    "1200 N Main St",
    "Bonham, TX 75418",
    "PISD PLANO EAST HS",
    "3000 LOS RIOS BLVD",
    "PLANO, TX 75074",
    "WRA Architects",
    "12377 Merit Dr"
]

redact_text_in_pdf(INPUT_PDF, OUTPUT_PDF, search_terms)

## Replace Text in the PDF

In [2]:
import fitz

def replace_text_in_pdf(input_pdf, output_pdf, replacements):
    """
    Replace text in PDF (works best with text, not scanned images)
    """
    doc = fitz.open(input_pdf)
    
    for page in doc:
        for old_text, new_text in replacements.items():
            text_instances = page.search_for(old_text)
            for inst in text_instances:
                # Remove old text
                page.add_redact_annot(inst)
                page.apply_redactions()
                
                # Add new text (approximate position)
                page.insert_text(
                    inst[:2],  # (x, y) from rect
                    new_text,
                    fontsize=10,
                    color=(0, 0, 0)
                )
    
    doc.save(output_pdf)
    doc.close()

replacements = {
    "Bonham ISD Additions and Renovations": "PROJECT A",
    "1200 N Main St, Bonham, TX 75418": "LOCATION REDACTED",
    "PISD PLANO EAST HS": "PROJECT B",
    "3000 LOS RIOS BLVD PLANO, TX 75074": "LOCATION REDACTED"
}

replace_text_in_pdf(INPUT_PDF, OUTPUT_PDF, replacements)

## Complete Anonymization Script

In [8]:
import fitz
import re

def anonymize_pdf(input_pdf, output_pdf, custom_replacements=None):
    """
    Complete anonymization: text redaction + metadata removal
    """
    doc = fitz.open(input_pdf)
    
    # Common patterns to redact
    patterns = [
        r'\d{3,5}\s+[A-Z][a-z]+.*(?:St|Ave|Rd|Dr|Blvd)',  # Addresses
        r'[A-Z]{2}\s+\d{5}',  # ZIP codes
        r'\d{3}[-.]?\d{3}[-.]?\d{4}',  # Phone numbers
    ]
    
    # Add custom terms
    if custom_replacements:
        patterns.extend(custom_replacements)
    
    for page in doc:
        # Get all text
        text = page.get_text()
        
        # Find and redact based on patterns
        for pattern in patterns:
            matches = re.finditer(pattern, text)
            for match in matches:
                # This is simplified - actual implementation needs
                # to map text position to PDF coordinates
                search_results = page.search_for(match.group())
                for rect in search_results:
                    page.add_redact_annot(rect, fill=(0, 0, 0))
        
        page.apply_redactions()
    
    # Remove metadata
    doc.set_metadata({})
    
    # Save with optimization
    doc.save(
        output_pdf,
        garbage=4,      # Remove unused objects
        deflate=True,   # Compress
        clean=True      # Clean up
    )
    doc.close()

additional_replacements = {
    "Bonham ISD Additions and Renovations": "PROJECT A",
    "1200 N Main St, Bonham, TX 75418": "LOCATION REDACTED",
    "PISD PLANO EAST HS": "PROJECT B",
    "3000 LOS RIOS BLVD PLANO, TX 75074": "LOCATION REDACTED",
    "WRA Architects": "ARCHITECTs Co."
}

anonymize_pdf(INPUT_PDF, OUTPUT_PDF, additional_replacements)